# 00 · De palabras a números

**Duración estimada:** 20 minutos

## Propósito

Una computadora no sabe leer. Sabe hacer cuentas.

Entonces, para que una computadora pueda comparar dos textos, primero hay que
convertir esos textos en números. Este cuaderno hace esa conversión **a mano**,
con listas y diccionarios de Python, sin ninguna librería especializada.

Es el paso que después van a hacer las librerías por nosotros. Conviene haberlo
hecho una vez a mano para saber qué están haciendo.

## Qué vamos a necesitar

Nada. Solo Python.

## 1. El problema

Tenemos cuatro oraciones. Queremos que la computadora nos diga cuáles se parecen
entre sí.

In [ ]:
documentos = [
    "El perro muerde al hombre",
    "El hombre muerde al perro",
    "El perro come carne",
    "El hombre come comida",
]

numero = 0
for documento in documentos:
    numero = numero + 1
    print("Documento", numero, ":", documento)

Vos podés leerlos y decir cuáles se parecen. La computadora no.

Para ella, `"El perro come carne"` es una secuencia de veinte caracteres. No sabe
dónde termina una palabra, y no sabe que *perro* aparece también en otro documento.

Vamos a arreglar eso en tres pasos: **tokenizar**, **armar el vocabulario** y
**contar**.

## 2. Primer paso: tokenizar

**Token** es la unidad mínima de análisis. Por ahora, una palabra.

Tokenizar es partir el texto en tokens. La forma más simple es cortar por los
espacios, y como además queremos que *El* y *el* cuenten como la misma palabra,
pasamos todo a minúsculas.

In [ ]:
# Guardamos, para cada documento, su lista de tokens.
documentos_en_tokens = []

for documento in documentos:
    en_minusculas = documento.lower()
    tokens = en_minusculas.split()
    documentos_en_tokens.append(tokens)

numero = 0
for tokens in documentos_en_tokens:
    numero = numero + 1
    print("Documento", numero, ":", tokens)

> **Ya tomamos una decisión.** Pasar todo a minúsculas es una decisión de
> análisis, no un paso técnico obligatorio. Estamos afirmando que *El* y *el* son
> la misma palabra.
>
> En estos cuatro documentos parece obvio. En un corpus real no lo es: si
> pasamos todo a minúsculas, *Mercedes* la ciudad y *mercedes* el plural de
> merced se vuelven la misma cosa, y perdemos la posibilidad de reconocer nombres
> propios.
>
> Guardá esta decisión. Volvemos sobre ella en el práctico.

## 3. Segundo paso: el vocabulario

El **vocabulario** es la lista de todas las palabras distintas que aparecen en el
conjunto de documentos. Es la misma para todos: es la regla común que permite
comparar.

Lo armamos recorriendo todos los tokens y quedándonos con los que no vimos antes.

In [ ]:
vocabulario = []

for tokens in documentos_en_tokens:
    for token in tokens:
        if token not in vocabulario:
            vocabulario.append(token)

# Lo ordenamos alfabéticamente para que el orden no dependa
# de en qué documento apareció cada palabra primero.
vocabulario.sort()

print("palabras distintas:", len(vocabulario))
print(vocabulario)

Ocho palabras distintas en cuatro oraciones.

Cada posición de esa lista va a ser una **columna**. La palabra `al` va a estar
siempre en la posición 0, `carne` en la 1, y así.

## 4. Tercer paso: contar

Ahora, para cada documento, contamos cuántas veces aparece cada palabra del
vocabulario. El resultado es una lista de ocho números: un número por columna.

Esa lista de números es el **vector** del documento.

In [ ]:
# Empezamos con un solo documento, para ver el mecanismo.
tokens_del_primero = documentos_en_tokens[0]

vector = []
for palabra in vocabulario:
    veces = tokens_del_primero.count(palabra)
    vector.append(veces)

print("documento:", documentos[0])
print()
print("vocabulario:", vocabulario)
print("vector     :", vector)

Leelo despacio: el `1` de la posición 0 dice que `al` aparece una vez. El `0`
de la posición 1 dice que `carne` no aparece. Y así.

**Ese vector es el documento, para la computadora.** Ya no es texto: son ocho
números que se pueden sumar, restar y comparar.

Hagamos lo mismo con los cuatro.

In [ ]:
vectores = []

for tokens in documentos_en_tokens:
    vector = []
    for palabra in vocabulario:
        veces = tokens.count(palabra)
        vector.append(veces)
    vectores.append(vector)

numero = 0
for vector in vectores:
    numero = numero + 1
    print("Documento", numero, ":", vector)

## 5. La tabla

Esos cuatro vectores, puestos uno debajo del otro, forman una tabla: las columnas
son las palabras del vocabulario y las filas son los documentos.

Se la llama **matriz documento-término**, y es el objeto central de toda esta
unidad.

In [ ]:
# pandas: para mostrar la tabla. Ya la usaron en la unidad anterior.
import pandas as pd

nombres_de_fila = []
numero = 0
for documento in documentos:
    numero = numero + 1
    nombres_de_fila.append("Doc " + str(numero))

tabla = pd.DataFrame(vectores, columns=vocabulario, index=nombres_de_fila)
tabla

## 6. Lo que ya se puede hacer con esto

Una tabla de números permite preguntas que un texto no permite.

**¿Qué palabra es la más frecuente en todo el conjunto?** Es sumar cada columna.

In [ ]:
suma_por_palabra = tabla.sum()

print(suma_por_palabra.sort_values(ascending=False))

**¿Cuántas palabras distintas tiene cada documento?** Es contar cuántas
columnas de esa fila son distintas de cero.

In [ ]:
for nombre in nombres_de_fila:
    fila = tabla.loc[nombre]
    palabras_presentes = 0
    for valor in fila:
        if valor > 0:
            palabras_presentes = palabras_presentes + 1
    print(nombre, "->", palabras_presentes, "palabras distintas")

**¿Qué dos documentos se parecen más?** Se pueden comparar los vectores
posición por posición y contar en cuántas coinciden.

Miremos primero los documentos 1 y 2.

In [ ]:
vector_1 = vectores[0]
vector_2 = vectores[1]

print("Doc 1:", documentos[0])
print("       ", vector_1)
print()
print("Doc 2:", documentos[1])
print("       ", vector_2)
print()
print("¿son iguales?", vector_1 == vector_2)

**Son idénticos.**

*El perro muerde al hombre* y *El hombre muerde al perro* tienen exactamente el
mismo vector, porque usan exactamente las mismas palabras.

Y no son lo mismo. En periodismo, de hecho, la diferencia entre esas dos
oraciones es el ejemplo clásico de qué es una noticia.

**Al convertir el texto en una bolsa de palabras perdimos el orden**, y con el
orden perdimos quién le hace qué a quién. Es la limitación más importante de todo
lo que vamos a ver hoy, y conviene tenerla presente desde el principio.

> ### Perilla
>
> En el paso 2 decidimos tokenizar cortando por espacios. Probá qué pasa con un
> texto que tenga puntuación.

In [ ]:
prueba = "El perro come carne. El gato, no."

print("cortando por espacios:")
print(prueba.lower().split())

`carne.` con el punto pegado y `gato,` con la coma quedaron como palabras
propias. Para la computadora, `carne.` y `carne` son dos entradas distintas del
vocabulario.

Ese es el problema que resuelve la **limpieza**, y es el primer tema del práctico.

## Lo que hiciste hoy

- Convertiste texto en números, a mano, en tres pasos: tokenizar, armar el
  vocabulario, contar.
- Construiste una matriz documento-término.
- Viste que dos oraciones con distinto significado pueden tener el mismo vector.
- Tomaste dos decisiones de análisis sin darte cuenta: pasar a minúsculas y
  cortar por espacios.

## Lo que viene

En el `01` hacemos lo mismo con una librería, en tres líneas. La pregunta va a
ser si esas tres líneas hacen exactamente lo que acabás de hacer vos.